# 機械学習で見つける ── 作業日誌・写真・糖度・集落のデータで「機械学習の基礎」を歩き直す

画面：`pages/16_🧠_機械学習で見つける.py`／部品：`ml_insight.py`（numpyだけで書いた決定木・PCA・k-means・重回帰・DID）。
このノートでは、**部品の答えが授業と同じ scikit-learn の答えと一致するか**を確かめます（第2〜4回：AIのコードは動いても正しいとは限らない）。

> Colab ではリポジトリを clone してから実行します。

In [ ]:
# !git clone https://github.com/nayukaya3-cn/Smart-Bakery-Farmer_20260923.git
# %cd Smart-Bakery-Farmer_20260923
import numpy as np, pandas as pd
import ml_insight as ml
DATA = 'data/'

## ① 前処理（第5〜6回）：本文 → 特徴量、正解 → 0/1

In [ ]:
d = pd.read_csv(DATA+'sample_work_diary.csv', encoding='utf-8-sig').merge(
    pd.read_csv(DATA+'sample_work_diary_truth.csv', encoding='utf-8-sig'), on='ID')
F = ml.features(d)
y = ml.truth_vector(d['困りごと'], hold='除く')
m = y.notna()
X, yy = F[m].to_numpy(float), y[m].astype(int).to_numpy()
F[m].assign(正解=yy).head()

## ② 判別（第7〜10回）：決定木を自作版と scikit-learn で比べる

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, f1_score
tree = ml.fit_tree(X, yy, max_depth=2)
mine = (ml.predict_proba(tree, X) >= 0.5).astype(int)
sk = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, yy).predict(X)
print('一致した件数:', (mine == sk).sum(), '/', len(yy))
print('自作の混同行列:', ml.confusion(yy, mine))
print('sklearn:', confusion_matrix(yy, sk).ravel(), '(TN, FP, FN, TP)')
print('\n'.join(ml.tree_rules(tree, list(F.columns))))

**問い**：木は「文字数」を使っていますか？ 「特に問題なし。」が短いことを使った近道（ショートカット学習）です。`文字数` の列を落として F値の変化を確かめましょう。

In [ ]:
Xn = F[m].drop(columns='文字数').to_numpy(float)
for name, XX in [('全特徴量', X), ('文字数なし', Xn)]:
    p = ml.cross_val_predict(XX, yy, k=5, max_depth=2)
    print(name, '交差検証のF値:', round(ml.scores(ml.confusion(yy, (p >= .5).astype(int)))['F値'], 3))
print('人の規則のF値:', round(ml.scores(ml.confusion(yy, ml.rule_handmade(F[m])))['F値'], 3))

## ✅ トレース表（DNCL）：混同行列を1件ずつ数える

In [ ]:
ml.trace_confusion(yy[:8], mine[:8], d.loc[m, 'ID'].head(8).tolist())

## ③ 教師なし（第13〜14回）：写真のタグ → z標準化 → PCA → k-means

In [ ]:
from sklearn.decomposition import PCA
ph = pd.read_csv(DATA+'sample_photo_ai.csv', encoding='utf-8-sig')
ph = ph[ph['種類'] == '現場写真'].reset_index(drop=True)
M = ml.photo_matrix(ph); M = M.loc[:, M.sum() > 0]
Z = ml.zscore(M.to_numpy())
print('自作PCAの寄与率:', ml.pca(Z)['ratio'].round(3))
print('sklearn PCAの寄与率:', PCA(2).fit(Z).explained_variance_ratio_.round(3))
print(ml.elbow(Z, range(1, 6)))
km = ml.kmeans(Z, 3)
M.assign(クラスター=km['labels']).groupby('クラスター').mean().round(2).T.sort_values(0, ascending=False).head(10)

## ④ 回帰（第11〜12回）：糖度 ~ 週 + 寒さ + 週×寒さ（最後の測定日をテストに）

In [ ]:
from sklearn.linear_model import LinearRegression
b = pd.read_csv(DATA+'sample_spinach_brix.csv', encoding='utf-8-sig')
b['週'] = (pd.to_datetime(b['測定日']) - pd.to_datetime(b['測定日']).min()).dt.days / 7
b['寒さ'] = (b['区'] == '寒さに当てる区').astype(int); b['週×寒さ'] = b['週'] * b['寒さ']
xs = ['週', '寒さ', '週×寒さ']; last = b['測定日'].max()
tr, te = b[b['測定日'] != last], b[b['測定日'] == last]
beta = ml.fit_ols(tr[xs].to_numpy(), tr['糖度'])
lr = LinearRegression().fit(tr[xs], tr['糖度'])
print('自作:', beta.round(3)); print('sklearn:', np.r_[lr.intercept_, lr.coef_].round(3))
print(ml.reg_scores(te['糖度'], ml.predict_ols(beta, te[xs].to_numpy())))

## ⑤ 相関と因果（第15回）：差の差（DID）

In [ ]:
r = ml.did(pd.read_csv(DATA+'sample_shuraku_did.csv', encoding='utf-8-sig'), '集落', '年', '交付金', '経営耕地_ha')
print({k: round(v, 2) for k, v in r.items() if isinstance(v, float)})
r['表']

**3行の報告**：観測（数えた事実・単位と分母）／解釈（根拠のIDと原文）／限界（件数・正解の付け方・平行トレンド）に分けて書きましょう。